#### 중첩 for문 대신 generator 활용

#### Comprehension, Assignment Expression

In [ ]:
import re

from typing import Iterable, Set

def collect_account_ids_from_arns(arns):
    matched_arns = filter(None, (re.match(ARN_REGEX, arn) for arn in arns))
    return {m.groupdict()['account_id'] for m in matched_arns}

def collect_account_ids_from_arns(arns: Iterable[str]) -> Set[str]:
    return {
        matched.groupdict()['account_id']
        for arn in arns
        if (matched := re.match(ARN_REGEX, arn)) is not None
    }

#### container iterable (p.74)
- \_\_iter__ 내에 generator 사용
- 인스턴스가 호출될 때마다 새로운 generator 를 리턴함
- next 를 정의하지 않기 때문에, iteratozr는 아님 (iterable)

In [1]:
from datetime import timedelta

class DateRangeContainerIterable:
    def __init__(self, start_date, end_date):
        self.start_date = start_date
        self.end_date = end_date

    def __iter__(self):
        current_day = self.start_date
        while current_day < self.end_date:
            yield current_day
            current_day += timedelta(days=1)

In [2]:
from datetime import date

r = DateRangeContainerIterable(date(2022, 1, 1), date(2022, 1, 5))
next(r)

TypeError: 'DateRangeContainerIterable' object is not an iterator

In [3]:
for r in DateRangeContainerIterable(date(2022, 1, 1), date(2022, 1, 5)):
    print(r)

2022-01-01
2022-01-02
2022-01-03
2022-01-04


#### container Object (p.77)
- \_\_contains__ method
- Boolean 값 반환
- in 키워드에서 호출
- 가독성 높은 코드가 됨

In [ ]:
class Boundaries:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def __contains__(self, coord):
        x, y = coord
        return 0 <= x < self.width and 0 <= y < self.height
    
class Grid:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.limits = Boundaries(width, height)

    def __contains__(self, coord):
        return coord in self.limits
    
def mark_coordinate(grid, coord):
    if coord in grid:
        grid[coord] = MARKED

#### \_\_getattr__ 속성
- 객체 속성 접근 순서
  1. \_\_dict__ 사전에 존재 시 - \_\_getattribute__ 메서드 호출
  2. 존재하지 않는 속성인 경우, \_\_getattr__ 메서드 호출

In [4]:
class DynamicAttributes:
    def __init__(self, attribute):
        self.attribute = attribute

    def __getattr__(self, attr):
        if attr.startswith('fallback_'):
            name = attr.replace('fallback_', '')
            return f'[fallback resolved] {name}'
        raise AttributeError(f'In {self.__class__.__name__}, {attr} attribute does not exist')

In [5]:
dyn = DynamicAttributes('value')

In [6]:
dyn.attribute

'value'

In [12]:
dyn.fallback_test = 'abcd'
dyn.fallback_test

'abcd'

In [9]:
dyn.fallback_test2

'[fallback resolved] test2'

In [10]:
dyn.__dict__

{'attribute': 'value', 'fallback_test': 'abcd'}

In [11]:
getattr(dyn, 'something', 'default')

'default'

#### Callable
- 객체를 일반 함수처럼 호출하면 \_\_call__ 매직 메서드가 호출됨


In [14]:
from collections import defaultdict

class CallCount:
    def __init__(self):
        self._counts = defaultdict(int)

    def __call__(self, arg):
        self._counts[arg] += 1
        return self._counts[arg]
    
cc = CallCount()
cc(1)
cc(2)
cc(1)
cc(1)
cc._counts

defaultdict(int, {1: 3, 2: 1})

#### Magic Method Summary

- \_\_getItem__(key) - 첨자형 객체 (obj[key], obj[i:j], obj[i:j:k])
- \_\_enter__ / \_\_exit__ - Context Manager (with obj: ...)
- \_\_iter__ / \_\_next__ - Iterable object (for i in obj: ...)
- \_\_len__ / \_\_getItem__ - Iterable object (for i in obj: ...)
- \_\_getattr__ - 동적 속성 조회 (obj.\<attribute>)
- \_\_call__ - Callable object (obj(*args, **kwargs))



#### mixin
- 코드를 재사용하기 위해 일반적인 행동을 캡슐화해 놓은 부모 클래스

In [18]:
class BaseTokenizer:
    def __init__(self, str_token):
        self.str_token = str_token

    def __iter__(self):
        yield from self.str_token.split('-')

tk = BaseTokenizer("abc-def-ghi")
list(tk)

['abc', 'def', 'ghi']

In [19]:
class UpperIterableMixin:
    def __iter__(self):
        return map(str.upper, super().__iter__())
    
class Tokenizer(UpperIterableMixin, BaseTokenizer):
    pass

In [20]:
[cls.__name__ for cls in Tokenizer.mro()]

['Tokenizer', 'UpperIterableMixin', 'BaseTokenizer', 'object']

In [22]:
tk = Tokenizer("abc-def-ghi")
list(tk)

['ABC', 'DEF', 'GHI']

In [27]:
tk.__dict__

{'str_token': 'abc-def-ghi'}

#### unpacking

In [28]:
first, *rest = range(6)
print(first)
print(rest)

0
[1, 2, 3, 4, 5]


In [29]:
first, last, *empty = 1, 2
print(first)
print(last)
print(empty)

1
2
[]


In [32]:
def f(x, y):
    print(f'{x=} {y=}')

f(1,2)

x=1 y=2


In [39]:
next((i for i in range(0)), -1)

-1

In [38]:
g = (i for i in range(0))
next(g)

StopIteration: 